In [0]:
# Programmatic Parameter Configuration leveraging Modern Volumes instead of DBFS root
dbutils.widgets.text("input_root_path", "/Volumes/workspace/default/takehome_raw_files", "Input Volume Path")
dbutils.widgets.text("catalog_db_name", "client_analytics_th", "Target Database Name")

INPUT_ROOT = dbutils.widgets.get("input_root_path")
DB_NAME = dbutils.widgets.get("catalog_db_name")

# Recreate the environment isolated under Unity Catalog
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
spark.sql(f"USE {DB_NAME}")

print(f"Pipeline initialized.\nTarget Database: {DB_NAME}\nInput Volume Path: {INPUT_ROOT}")


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import pyspark.sql.functions as F

# Client Source Configurations mapping raw parameters to expected inputs
# Reflects the exact filenames specified in the assignment document
CLIENT_CONFIGS = {
    "client_a": {
        "customers": {"delimiter": ",", "file_pattern": "customers_2024_01.txt"},
        "transactions": {"delimiter": ",", "file_pattern": "transactions_2024_01.txt"}
    },
    "client_b": {
        "customers": {"delimiter": "|", "file_pattern": "customer_dump.txt"},
        "transactions": {"delimiter": "|", "file_pattern": "txn_history.txt"}
    },
    "client_c": {
        "customers": {"delimiter": "\t", "file_pattern": "customers.txt"},
        "transactions": {"delimiter": "\t", "file_pattern": "transactions.txt"}
    }
}

In [0]:
def ingest_to_bronze(client_id, entity_type, config):
    """Ingests raw text files from Unity Catalog Volumes into Delta Bronze tables safely."""
    # Constructing paths to match the folder structure created within the Managed Volume
    path = f"{INPUT_ROOT}/{client_id}/{config['file_pattern']}"
    
    try:
        # Read everything as StringType to prevent dropouts over corrupt schemas or bad data types
        raw_df = spark.read \
            .option("delimiter", config["delimiter"]) \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .csv(path)
        
        #raw_df.display()

        #print(raw_df)
        
        # Inject metadata lineage fields directly at point of capture
        # FIXED: Replaced legacy F.input_file_name() with UC-compliant F.col("_metadata.file_path")
        bronze_df = raw_df \
            .withColumn("source_client", F.lit(client_id)) \
            .withColumn("ingested_at", F.current_timestamp()) \
            .withColumn("input_file_name", F.col("_metadata.file_path"))
        
        table_name = f"bronze_{client_id}_{entity_type}"
        
        # Write to Delta table format
        bronze_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
        print(f"Successfully materialized Bronze table: {table_name}")
    except Exception as e:
        print(f"❌ Failed processing for {client_id} {entity_type}. Error details: {str(e)}")

# Execute Bronze Ingestion Engine
for client, entities in CLIENT_CONFIGS.items():
    for entity, cfg in entities.items():
        ingest_to_bronze(client, entity, cfg)

In [0]:
def build_gold_dimensions_and_facts():
    """Generates globally unified dimensions and facts using reliable surrogate keys[cite: 1]."""
    
    # Dim Customers
    silver_cust = spark.read.table("silver_customers")
    gold_customers = silver_cust \
        .withColumn("customer_sk", F.sha2(F.concat_ws("||", F.col("source_client"), F.col("client_cust_id")), 256)) \
        .select(
            "customer_sk", "source_client", "client_cust_id", 
            "first_name", "last_name", "email", "ingested_at"
        ).dropDuplicates(["customer_sk"])
        
    gold_customers.write.format("delta").mode("overwrite").saveAsTable("dim_customers")
    
    # Fact Transactions
    silver_txn = spark.read.table("silver_transactions")
    gold_transactions = silver_txn \
        .withColumn("transaction_sk", F.sha2(F.concat_ws("||", F.col("source_client"), F.col("client_txn_id")), 256)) \
        .withColumn("customer_sk", F.sha2(F.concat_ws("||", F.col("source_client"), F.col("client_cust_id")), 256)) \
        .select(
            "transaction_sk", "customer_sk", "source_client", "client_txn_id",
            "amount", "transaction_timestamp", "ingested_at"
        )
        
    gold_transactions.write.format("delta").mode("overwrite").saveAsTable("fct_transactions")
    print("Gold Analytic Dimensions and Facts successfully generated[cite: 1]!")

# Execute Gold Generation
build_gold_dimensions_and_facts()

In [0]:
def process_silver_customers():
    """Maps and cleans customer schemas to an intermediate, uniform Silver representation.
    Dynamically handles structural variations like combined 'full_name' or 'fname'/'lname' columns.
    """
    
    # -------------------------------------------------------------------------
    # CLIENT A: Expects standard native customer fields
    # -------------------------------------------------------------------------
    a_source = spark.read.table("bronze_client_a_customers")
    a_cols = a_source.columns
    a_cust_col = "customer_id" if "customer_id" in a_cols else "id"
    
    # FIXED: Replaced F.try_cast with F.expr("try_cast(...)") for syntax compatibility
    a_df = a_source.select(
        F.expr(f"try_cast({a_cust_col} AS string)").alias("client_cust_id"),
        F.col("first_name"), 
        F.col("last_name"),
        F.col("email"), 
        F.col("source_client"), 
        F.col("ingested_at")
    )
        
    # -------------------------------------------------------------------------
    # CLIENT B: Features 'cust_id' and a combined 'full_name' column
    # -------------------------------------------------------------------------
    b_source = spark.read.table("bronze_client_b_customers")
    b_cols = b_source.columns
    b_cust_col = "cust_id" if "cust_id" in b_cols else "customer_id"
    
    if "full_name" in b_cols:
        # Clean the full_name column by replacing pipes with spaces
        cleaned_full_name = F.regexp_replace(F.col("full_name"), "\\|", " ")
        
        # Use the cleaned reference for your substring logic
        derived_first = F.substring_index(cleaned_full_name, " ", 1)
        derived_last = F.when(
            F.locate(" ", cleaned_full_name) > 0, 
            F.substring_index(cleaned_full_name, " ", -1)
        ).otherwise(F.lit(""))

        # derived_first = F.substring_index(F.col("full_name"), " ", 1)
        # derived_last = F.when(
        #     F.locate(" ", F.col("full_name")) > 0, 
        #     F.substring_index(F.col("full_name"), " ", -1)
        # ).otherwise(F.lit("")) 
    else:
        derived_first = F.col("first_name") if "first_name" in b_cols else F.lit(None).cast("string")
        derived_last = F.col("last_name") if "last_name" in b_cols else F.lit(None).cast("string")

    b_df = b_source.select(
        F.expr(f"try_cast({b_cust_col} AS string)").alias("client_cust_id"),
        derived_first.alias("first_name"),
        derived_last.alias("last_name"),
        F.col("contact_email").alias("email"),
        F.col("source_client"), 
        F.col("ingested_at")
    )
        
    # -------------------------------------------------------------------------
    # CLIENT C: Resolves 'fname' / 'lname' and missing emails
    # -------------------------------------------------------------------------
    c_source = spark.read.table("bronze_client_c_customers")
    c_cols = c_source.columns
    c_cust_col = "id" if "id" in c_cols else "customer_id"
    
    derived_c_first = F.col("fname") if "fname" in c_cols else (F.col("first_name") if "first_name" in c_cols else F.lit(None).cast("string"))
    derived_c_last = F.col("lname") if "lname" in c_cols else (F.col("last_name") if "last_name" in c_cols else F.lit(None).cast("string"))
    
    c_df = c_source.select(
        F.expr(f"try_cast({c_cust_col} AS string)").alias("client_cust_id"),
        derived_c_first.alias("first_name"),
        derived_c_last.alias("last_name"),
        F.col("email") if "email" in c_cols else F.lit(None).cast("string").alias("email"), 
        F.col("source_client"), 
        F.col("ingested_at")
    )
    
    # Unifying variations into a clear operational dataset
    silver_customers = a_df.unionByName(b_df).unionByName(c_df)
    
    # Structural Cleaning: Drops records missing core keys
    cleaned_silver = silver_customers.filter(F.col("client_cust_id").isNotNull())
    cleaned_silver.write.format("delta").mode("overwrite").saveAsTable("silver_customers")
    print("Silver Customers Layer materialized successfully.")


def process_silver_transactions():
    """Maps, filters, and standardizes transaction data types across sources.
    Uses robust SQL try_cast expressions to protect pipelines from parsing failures on malformed data.
    """
    
    # -------------------------------------------------------------------------
    # CLIENT A: Resolves 'created_ts' variance vs expected 'transaction_date'
    # -------------------------------------------------------------------------
    a_source = spark.read.table("bronze_client_a_transactions")
    a_cols = a_source.columns
    a_date_col = "created_ts" if "created_ts" in a_cols else "transaction_date"
    a_amt_col = "amount" if "amount" in a_cols else "total"
    
    a_txn = a_source.select(
        F.col("transaction_id").alias("client_txn_id"),
        F.col("customer_id").alias("client_cust_id"),
        F.expr(f"try_cast({a_amt_col} AS double)").alias("amount"),
        F.expr(f"try_cast({a_date_col} AS timestamp)").alias("transaction_timestamp"),
        F.col("source_client"), 
        F.col("ingested_at")
    )
    
    # -------------------------------------------------------------------------
    # CLIENT B: Resolves 'total' metric and 'txn_ts' timestamp configuration
    # -------------------------------------------------------------------------
    b_source = spark.read.table("bronze_client_b_transactions")
    b_cols = b_source.columns
    b_date_col = "txn_ts" if "txn_ts" in b_cols else ("txn_date" if "txn_date" in b_cols else "transaction_date")
    b_amt_col = "total" if "total" in b_cols else "tx_amount"
    
    b_txn = b_source.select(
        F.col("txn_id").alias("client_txn_id"),
        F.col("cust_id").alias("client_cust_id"),
        F.expr(f"try_cast({b_amt_col} AS double)").alias("amount"),
        F.expr(f"try_cast({b_date_col} AS timestamp)").alias("transaction_timestamp"),
        F.col("source_client"), 
        F.col("ingested_at")
    )

    # -------------------------------------------------------------------------
    # CLIENT C: Resolves missing timestamp column & handles currency text cleans
    # -------------------------------------------------------------------------
    c_source = spark.read.table("bronze_client_c_transactions")
    c_cols = c_source.columns
    
    c_amt_expr = "final_amount" if "final_amount" in c_cols else ("price_usd" if "price_usd" in c_cols else "total")
    # First apply string transformations to remove formatting characters like '$'
    cleaned_c_amount_str = F.regexp_replace(F.col(c_amt_expr), r'[\$,]', '')
    
    c_cust_col = "user_id" if "user_id" in c_cols else "customer_id"
    
    # Inline evaluation of safe fallback for missing column strategy
    if "created_at" in c_cols:
        derived_c_date_expr = "try_cast(created_at AS timestamp)"
    elif "transaction_date" in c_cols:
        derived_c_date_expr = "try_cast(transaction_date AS timestamp)"
    else:
        derived_c_date_expr = "ingested_at"
    
    c_txn = c_source.select(
        F.col("id").alias("client_txn_id"),
        F.col(c_cust_col).alias("client_cust_id"),
        F.expr(f"try_cast({c_amt_expr} AS double)").alias("amount"), # try_cast automatically returns null on values like '$336.51' if they slipped through, but let's cast our cleaned version explicitly:
        F.col("source_client"), 
        F.col("ingested_at")
    )
    
    # Let's adjust Client C's selection to cleanly reference the regex column expression object
    c_txn = c_source.select(
        F.col("id").alias("client_txn_id"),
        F.col(c_cust_col).alias("client_cust_id"),
        # Use expr string interpolation cleanly on the underlying transformations
        F.expr(f"try_cast(regexp_replace({c_amt_expr}, '[\\\\$,]', '') AS double)").alias("amount"),
        F.expr(derived_c_date_expr).alias("transaction_timestamp"),
        F.col("source_client"), 
        F.col("ingested_at")
    )
    
    # Unifying all processed transactional entities safely
    silver_txns = a_txn.unionByName(b_txn).unionByName(c_txn)
    
    # Data Quality Validation: Filter out rows where try_cast returned NULL due to corruption (like 'EXTRA')
    valid_txns = silver_txns.filter((F.col("amount").isNotNull()) & (F.col("client_txn_id").isNotNull()))
    valid_txns.write.format("delta").mode("overwrite").saveAsTable("silver_transactions")
    print("Silver Transactions Layer materialized successfully.")

# Run Silver Pipeline Transforms
process_silver_customers()
process_silver_transactions()